In [ ]:
import json
from pathlib import Path

import yaml
from transformers import AutoTokenizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = yaml.safe_load((ROOT / "config/rag.yaml").read_text(encoding="utf-8"))
emb = cfg["embedding"]

tok = AutoTokenizer.from_pretrained(emb["model"], revision=emb["revision"])
chunks = [json.loads(line) for line in (ROOT / cfg["corpus"]["chunks"]).read_text(encoding="utf-8").splitlines()]

# Token count per chunk with the bge-m3 tokenizer; chunks above max_tokens would need splitting.
counts = sorted(((len(tok(c["text"])["input_ids"]), c["chunk_id"]) for c in chunks), reverse=True)
print("chunks:", len(counts))
print("median tokens:", counts[len(counts) // 2][0])
print("longest 10:", counts[:10])
print("over the limit:", [c for c in counts if c[0] > emb["max_tokens"]])